<a href="https://colab.research.google.com/github/Giomora-IA/IA--NOTA-AND-EJERCICIOS/blob/main/Alumno_ClasificacionSpotifyMLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clasificación del género musical con una red neuronal multicapa (Spotify Dataset)

En este ejercicio utilizaremos una red neuronal multicapa para resolver un problema de clasificación multiclase


Queremos resolver:

> **¿Podemos identificar el género musical de una canción a partir de sus características de audio?**

# Conjunto Datos

Utilizaremos algunas de las características musicales proporcionadas por Spotify y construiremos una red neuronal capaz de clasificar una canción dentro de uno de cinco géneros:

* Pop
* Rock
* Jazz
* Classical
* Hip-hop

La base de datos de Spotify, contiene información de miles de canciones disponibles en la plataforma Spotify.

Cada canción incluyen atributos cuantitativos como

* Danceability (bailabilidad),

* Energy (nivel de energía),

* Loudness (volumen promedio en decibelios),

* Speechiness (porcentaje de contenido hablado),

* Acousticness (nivel de acústica),

* Instrumentalness (nivel instrumental de la canción),

* Valence (positividad emocional),

* Tempo (ritmo de una canción, se mide en beats por minuto),

* Popularity (nivel de popularidad de la canción),

* Duration_ms (duración en milisegundos)

Entre otros.

Cargamos el dataset

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/mevangelista-alvarado/datasets/refs/heads/main/spotify_songs.csv"
df = pd.read_csv(url)

Visualizamos el dataframe

In [ ]:
df

In [ ]:
df.shape

# Conjunto de entrenamiento y test

## Explorar los datos

Numero de filas y columnas

In [ ]:
df.shape


Observamos informacion del dataframe

In [ ]:
df.info()

También podemos observar algunas estadísticas:


In [ ]:
df.describe()


In [ ]:
df.describe(include="object")

## Exploramos la etiqueta (label)

La variable que queremos predecir será:

```text
track_genre
```

Podemos observar los géneros disponibles:



In [ ]:
df['track_genre'].unique()

Y contar cuántas canciones existen para cada género:


In [ ]:
df['track_genre'].value_counts()


Notamos que nuestro conjunto de datos contiene muchos géneros diferentes.

Para este ejercicio utilizaremos únicamente:


In [ ]:
generos = [
    'pop',
    'rock',
    'jazz',
    'classical',
    'hip-hop'
]


Ahora filtramos el DataFrame:



In [ ]:
df = df[df['track_genre'].isin(generos)].copy()


In [ ]:
df.shape

Podemos comprobar el resultado:



In [ ]:
df['track_genre'].value_counts()


In [ ]:
df.shape

## Exploramos el input

Ahora necesitamos decidir qué información utilizaremos para intentar identificar el género.

Utilizaremos:


In [ ]:
# @title
features = [
    'danceability',
    'energy',
    'loudness',
    'speechiness',
    'acousticness',
    'instrumentalness',
    'liveness',
    'valence',
    'tempo'
]


Estas variables representan diferentes características de la canción.

Podemos observarlas:



In [ ]:
df[features].head()


Algunas de estas características representan:

| Variable           | Descripción                                |
| ------------------ | ------------------------------------------ |
| `danceability`     | Qué tan adecuada es la canción para bailar |
| `energy`           | Nivel de energía                           |
| `loudness`         | Volumen promedio                           |
| `speechiness`      | Presencia de palabras habladas             |
| `acousticness`     | Probabilidad de que sea acústica           |
| `instrumentalness` | Probabilidad de que sea instrumental       |
| `liveness`         | Probabilidad de una interpretación en vivo |
| `valence`          | Positividad o carácter emocional           |
| `tempo`            | Velocidad en BPM                           |

Estas serán las variables de entrada de nuestra red neuronal.

## Construir X e y


Construiremos `X` utilizando nuestras características musicales:



In [ ]:
X = df[features]


In [ ]:
X.shape

Y utilizaremos `track_genre` como variable objetivo:



In [ ]:
import pandas as pd

y = df['track_genre']
# One-hot encoding
Y = pd.get_dummies(y).values


In [ ]:
y

In [ ]:
Y

In [ ]:
pd.get_dummies(y).columns

Podemos comprobar las dimensiones:


In [ ]:
print("X:", X.shape)
print("y:", Y.shape)


Conceptualmente tenemos:

```text
Características musicales
          ↓
          X
          ↓
     Red neuronal
          ↓
          Y
          ↓
     Género musical
```

## Dividir los datos


Utilizaremos el 80 % para entrenar y el 20 % para evaluar.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42,
    stratify=Y
)


El parámetro `stratify=y_encoded` intenta conservar la proporción de cada género tanto en entrenamiento como en prueba.

In [ ]:
print(X_train.shape)
print(X_test.shape)


Normalizar las características

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


Recuerde que `StandardScaler`, transforma las variables para que tengan aproximadamente:
* media = 0
* desviación estándar = 1

# Modelo (Red Neuronal)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

modelo = Sequential([
    Dense(64, input_shape=(X_train.shape[1],), activation='relu'),
    Dense(32, activation='relu'),
    Dense(5, activation='softmax'),
])


La función `softmax` convierte la salida en probabilidades.

Por ejemplo:

```text
classical → 0.02
hip-hop   → 0.15
jazz      → 0.08
pop       → 0.65
rock      → 0.10
```

La red considera que `pop` es la clase más probable porque tiene la probabilidad más alta.

In [ ]:
modelo.summary()

Esto nos permite conocer:

* número de capas,
* número de neuronas,
* número de parámetros entrenables.

Optimizador

In [ ]:
from tensorflow.keras.optimizers import Adam

# Tasa de aprendizaje deseada
learning_rate = 0.01
adam_optimizer = Adam(learning_rate=learning_rate)


In [ ]:
modelo.compile(
    optimizer=adam_optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Entrenamiento

In [ ]:
history = modelo.fit(
    X_train, y_train,
    epochs=10, batch_size=2,
    validation_data=(X_test, y_test)
)


Visualizamos la función de pérdida

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['loss'], label='Pérdida de entrenamiento')
plt.plot(history.history['val_loss'], label='Pérdida de validación')
plt.xlabel('Épocas')
plt.ylabel('Pérdida')
plt.legend()
plt.title('Función de pérdida durante el entrenamiento')
plt.show()


Visualizamos el accuracy

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'], label='Precision en el entrenamiento')
plt.plot(history.history['val_accuracy'], label='Precision en la validación')
plt.xlabel('Épocas')
plt.ylabel('Pérdida')
plt.legend()
plt.title('Función de pérdida durante el entrenamiento')
plt.show()


# Evaluacion

In [ ]:
loss, accuracy = modelo.evaluate(X_test, y_test)
print(f'Loss: {loss}, Accuracy: {accuracy}')


## Matriz de Confusión

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, recall_score

y_pred = modelo.predict(X_test)
# Convertir las predicciones a clases
y_pred_classes = np.argmax(y_pred, axis=1)
y_test_classes = np.argmax(y_test, axis=1)

# Calcular la matriz de confusión
cm = confusion_matrix(y_test_classes, y_pred_classes)
# Calcular la sensibilidad
sensitivity = recall_score(y_test_classes, y_pred_classes, average=None)


In [ ]:
class_names = pd.get_dummies(y).columns.tolist()

# Visualizar la matriz de confusión como una imagen de colores
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap='Blues')

# Mostrar la sensibilidad (recall) para cada clase
print('Sensitivity (Recall) for each class:')
for i in range(5):
    print(f'Class {i}: {sensitivity[i]}')


# Prediccion

In [ ]:
class_names = pd.get_dummies(y).columns.tolist()

predictions = modelo.predict(X_test)
# Obtener el índice de la clase predicha para cada punto de datos
predicted_classes = np.argmax(predictions, axis=1) # y_pred
# Obtenemos el índice de la clase real de y_test
actual_classes = np.argmax(y_test, axis=1) # y_real

comparison = pd.DataFrame({
  'Actual Class Index': actual_classes,
  'Predicted Class Index': predicted_classes
})
# Agregar columnas con los nombres reales y predichos de las flores utilizando iris.target_names
comparison['Actual Flower'] = [class_names[i] for i in actual_classes]
comparison['Predicted Flower'] = [class_names[i] for i in predicted_classes]
# Reordenar las columnas para una mejor visualización
comparison = comparison[['Actual Class Index', 'Actual Flower', 'Predicted Class Index', 'Predicted Flower']]
print(comparison.head(30))


Predicción manual del género de una canción

In [ ]:
class_names = pd.get_dummies(y).columns.tolist()

# Introducir los valores de una canción
danceability = 0.72
energy = 0.50
loudness = 4.0
speechiness = 0.03
acousticness = 0.5
instrumentalness = 0.3
liveness = 0.25
valence = 0.85
tempo = 100

# Crear un df
cancion_nueva = pd.DataFrame([{
    'danceability': danceability,
    'energy': energy,
    'loudness': loudness,
    'speechiness': speechiness,
    'acousticness': acousticness,
    'instrumentalness': instrumentalness,
    'liveness': liveness,
    'valence': valence,
    'tempo': tempo
}])

# Escalar los datos de entrada usando el mismo scaler que se usó para el entrenamiento
cancion_nueva_scaled = scaler.transform(
    cancion_nueva
)

# Realizar la predicción con el modelo
probabilidades = modelo.predict(
    cancion_nueva_scaled,
    verbose=0
)

# Obtener las probabilidades
probabilidades = probabilidades[0]
# np.argmax() encuentra la posición que contiene el valor más grande.
clase_predicha = np.argmax(probabilidades)
genero_predicho = class_names[clase_predicha]


# Obtener el nombre de la cancion
print(f"Género predicho: {genero_predicho}")

# Mostrar todas las probabilidades
print("\nProbabilidades por género:")
print("----------------------------------------")
for genero, probabilidad in zip(
    class_names,
    probabilidades
):
    print(
        f"{genero:10s}: {probabilidad:.2%}"
    )
